In [27]:
#-- Packages --#

#--- Operational ---#
import os
import sys 
import pandas as pd
import numpy as np
import geopandas as gpd
from pathlib import Path
import json
import re
import geopandas as gpd
from __future__ import annotations
from typing import Any, Mapping, Sequence
import yaml



#--- Visualisations ---#
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

import seaborn as sns

#-- Directories --#
nb_dir = Path.cwd()
REPO_ROOT = nb_dir.parent

data_dir = REPO_ROOT / 'data/'
docs_dir = REPO_ROOT / 'docs/'

analysis_dir = data_dir / 'processed' / 'analysis/'
fires_dir = analysis_dir / "national_canadian_fires"

app_dir = data_dir / 'processed' / 'app/'
cache_dir = data_dir / 'processed' / 'cached/'

if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

#--- Constants ---#
cached_fires = cache_dir / 'NBAC/Canada_fires_1990_2024.parquet'

In [28]:
#-- Helper Functions --#
def extract_max_year(path):
    """
    Identify shapefile with latest year of available data
    """
    years = re.findall(r"\d{4}", path.stem)
    return max(map(int, years)) if years else -1


In [29]:
# #-- Load Files --#

# # National Canada polygons (GeoJSON)
# print(f'Loading National fires GeoJSON...')
# fires_GeoJSON_path = analysis_dir / "national_canadian_fires/Canada_fires_1990_2024.geojson"
# fires_GeoJSON = gpd.read_file(fires_GeoJSON_path)
# print(f" National Canada fires GeoJSON loaded. {fires_GeoJSON.crs}\n")

# # National Canada polygons (Shapefile)

# shp_files = list(fires_dir.glob("*.shp"))

# if not shp_files:
#     raise FileNotFoundError(f"No shapefiles found in {fires_dir}\n")

# fires_path = max(shp_files, key=extract_max_year)

# print(f"Loading National fires shapefile... \n File name: {fires_path.name}")
# fire_stats = gpd.read_file(fires_path)
# print(f" National Canada Fires loaded. {fire_stats.crs}\n")

# print(f"Cache National fires shapefile as parquet")
# cached_path = cache_dir / f"{fires_path.name[:-4]}.parquet"
# try:
#     print(" Caching...")
#     fire_stats.to_parquet(cached_path, index=False)
#     print(f" Cache complete. \n Saved to: {cached_path}" )
# except Exception as e:
#     raise RuntimeError(f"National fires shapefile failed to export as parquet: {e}")

In [30]:

# Load cached files

if not cached_fires.is_file():
    raise FileNotFoundError(
        f"NBAC cached parquet not found: {cached_fires}\n"
        "Run the 'Load File' block to generate it."
    )

try:
    NBAC = gpd.read_parquet(cached_fires)
    print("NBAC Wildfires cached parquet file found.")
    print(" Loading cached fires...")
    print(f" NBAC fires loaded. CRS: {NBAC.crs}\n")
except Exception as e:
    raise RuntimeError(
        f"Failed to read cached parquet: {cached_fires}\n"
        f"Error: {e}"
    ) from e


# AvCan regions
# =================================================================

print(f'Loading AvCan regions parquet...')
avcan_path = app_dir / "Regions.parquet"
avcan_regions= gpd.read_parquet(avcan_path)
print(f" AvCan Regions loaded. CRS: {avcan_regions.crs}\n")

# AvCan Fires 
# =================================================================

print(f'Loading all AvCan fires shapefile...')
avcan_fires_file = analysis_dir / 'avalanche_canada/fires/AvCan_fires_1990_2024.shp'

avcan_fires = gpd.read_file(avcan_fires_file)
print(f' AvCan fires loaded. CRS: {avcan_fires.crs}\n')

# Stage A Burn Severity Patches
# =================================================================

print(f'Loading Stage A Burn Severity Patches parquet...')
stage_a_patches_path = app_dir / 'Stage_A2_Burn_Severity_Patches.parquet'

stage_a_patches = gpd.read_parquet(stage_a_patches_path)
print(f' Stage A patchesloaded. CRS: {avcan_fires.crs}\n')



NBAC Wildfires cached parquet file found.
 Loading cached fires...
 NBAC fires loaded. CRS: {"$schema": "https://proj.org/schemas/v0.7/projjson.schema.json", "type": "GeographicCRS", "name": "WGS 84", "datum_ensemble": {"name": "World Geodetic System 1984 ensemble", "members": [{"name": "World Geodetic System 1984 (Transit)"}, {"name": "World Geodetic System 1984 (G730)"}, {"name": "World Geodetic System 1984 (G873)"}, {"name": "World Geodetic System 1984 (G1150)"}, {"name": "World Geodetic System 1984 (G1674)"}, {"name": "World Geodetic System 1984 (G1762)"}, {"name": "World Geodetic System 1984 (G2139)"}, {"name": "World Geodetic System 1984 (G2296)"}], "ellipsoid": {"name": "WGS 84", "semi_major_axis": 6378137, "inverse_flattening": 298.257223563}, "accuracy": "2.0", "id": {"authority": "EPSG", "code": 6326}}, "coordinate_system": {"subtype": "ellipsoidal", "axis": [{"name": "Geodetic latitude", "abbreviation": "Lat", "direction": "north", "unit": "degree"}, {"name": "Geodetic longi

In [31]:
stage_a_patches.columns.tolist()

['ee_feature_id',
 'Aspect Coherence (R)',
 'Majority Cardinal Direction',
 'Aspect Label',
 'Aspect Mean (deg)',
 'Max Elevation (m)',
 'Mean Elevation (m)',
 'Min Elevation (m)',
 'Elevation Relief (m)',
 'FireID',
 'Unique Fire ID (gid)',
 'National Park',
 'Patch Area (ha)',
 'Patch Area (m2)',
 'Patch ID',
 'Region',
 'Scenario',
 'Mean Slope Degree',
 'Slope Mean Percentage',
 'Slope Std Dev (deg)',
 'Subregion',
 'Year',
 'geometry',
 'Unique Patch ID']

## Cleaning

In [32]:

RENAME = {

    # Shared columns
    "gid": "Unique_gid_ID",
    "fireid": "Fire ID",
    "year": "Year",
    "prov_terr" : "Province/Territory",
    "natpark": "National Park",
    "cause": "Cause",

    # NBAC columns
    "adj_ha": "Adjusted Burn Area (ha)",

    # AvCan columns
    "region":"Region",
    "subregion":"Subregion",
    "subreg_ha":"Subregion Area (ha)",
    "tot_adj_ha":"Adjusted Burn Area (ha)"

}

NBAC_stats = NBAC.rename(columns=RENAME).copy()
avcan_stats = avcan_fires.rename(columns=RENAME).copy()

NBAC_stats["Province/Territory"] = NBAC_stats["Province/Territory"].replace({"PE": "PC"})



### Column Creation

In [33]:
# NBAC
NBAC_stats["Adjusted Burn Area (ha)"] = pd.to_numeric(NBAC_stats["Adjusted Burn Area (ha)"], errors="coerce")
avcan_stats["Adjusted Burn Area (ha)"] = pd.to_numeric(avcan_stats["Adjusted Burn Area (ha)"], errors="coerce")



NBAC_cause = NBAC_stats["Cause"].astype(str).str.strip().str.lower()
NBAC_stats["Is_Natural"] = (NBAC_cause == "natural").astype("int64")
NBAC_stats["Is_Human"] = (NBAC_cause == "human").astype("int64")
NBAC_stats["Is_Undetermined"] = (NBAC_cause == "undetermined").astype("int64")



PROV_TERR_RENAME = {
    "AB": "Alberta",
    "BC": "British Columbia",
    "MB": "Manitoba",
    "NB": "New Brunswick",
    "NL": "Newfoundland and Labrador",
    "NS": "Nova Scotia",
    "NT": "Northwest Territories",
    "NU": "Nunavut",
    "ON": "Ontario",
    "PC": "Prince Edward Island", 
    "QC": "Quebec",
    "SK": "Saskatchewan",
    "YT": "Yukon",
}
NBAC_stats["Province/Territory_full"] = NBAC_stats["Province/Territory"].map(PROV_TERR_RENAME)

# AvCan
avcan_cause_col = avcan_stats["Cause"].astype(str).str.strip().str.lower()
avcan_stats["Is_Natural"] = (avcan_cause_col == "natural").astype("int64")
avcan_stats["Is_Human"] = (avcan_cause_col == "human").astype("int64")
avcan_stats["Is_Undetermined"] = (avcan_cause_col == "undetermined").astype("int64")

## Aggregations

In [34]:

area_yearly = (
    NBAC_stats
      .groupby(["Province/Territory", "Year"], dropna=False)
      .agg(
          fires_n=("Unique_gid_ID", "nunique"),        # unique fires
          rows_n=("Unique_gid_ID", "size"),            # number of records (optional)
          area_sum_ha=("Adjusted Burn Area (ha)", "sum"),     # total burned area
          area_mean_ha=("Adjusted Burn Area (ha)", "mean"),   # mean per-record area (see note below)
          natural_cause=("Is_Natural", "sum"),
          human_cause=("Is_Human", "sum"),
          undetermined_cause=("Is_Undetermined", "sum")
      )
      .reset_index()
)

nbac_yearly = (
    NBAC_stats
      .groupby("Year", dropna=False)
      .agg(
          fires_n=("Unique_gid_ID", "nunique"),        # unique fires
          rows_n=("Unique_gid_ID", "size"),            # number of records (optional)
          area_sum_ha=("Adjusted Burn Area (ha)", "sum"),     # total burned area
          area_mean_ha=("Adjusted Burn Area (ha)", "mean"),   # mean per-record area (see note below)
          natural_cause=("Is_Natural", "sum"),
          human_cause=("Is_Human", "sum"),
          undetermined_cause=("Is_Undetermined", "sum")
      )
      .reset_index()
)

nbac_yearly["natural_pct"] = (nbac_yearly["natural_cause"]/ nbac_yearly['fires_n'] ) * 100

nbac_yearly["human_pct"] = (nbac_yearly["human_cause"]/ nbac_yearly['fires_n']) * 100

nbac_yearly["undetermined_pct"] = (nbac_yearly["undetermined_cause"]/ nbac_yearly['fires_n'] ) * 100


In [35]:
den = area_yearly["rows_n"].replace(0, pd.NA)

area_yearly["natural_pct"] = (area_yearly["natural_cause"] / den) * 100
area_yearly["human_pct"] = (area_yearly["human_cause"] / den) * 100
area_yearly["undetermined_pct"] = (area_yearly["undetermined_cause"] / den) * 100


In [36]:
bc_yearly = area_yearly[area_yearly["Province/Territory"] == "BC"].copy()


## Visualisations

### Canada

In [37]:
fig = go.Figure()

fig.update_layout(
    title_text = "Canadian Wildfires and Causes Over Time",
    xaxis_title = "Years",
    yaxis_title = "Count"
)

fig.add_trace( go.Scatter(
    x = nbac_yearly["Year"],
    y = nbac_yearly["fires_n"],
    name = "Canadian Wildfires",
    showlegend = True,

    customdata = (np.column_stack([
        nbac_yearly["natural_pct"],
        nbac_yearly["human_pct"],
        nbac_yearly["undetermined_pct"]
    ])),

    hovertemplate = (
        "Year: %{x}"
        "<br>Count: %{y}"
        "<br>Natural Cause: %{customdata[0]:.2f} %"
        "<br>Human Cause: %{customdata[1]:.2f} %"
        "<br>Undetermined Cause: %{customdata[2]:.2f} %"
    )

))

fig.add_trace(go.Scatter(
    x = nbac_yearly["Year"],
    y = nbac_yearly['natural_cause'],
    name = "Natural Cause",

    customdata = (
        nbac_yearly["natural_pct"]),


    hovertemplate = (
        "Year: %{x}"
        "<br>Count: %{y}"
        "<br>Natural Cause: %{customdata:.2f} %"
    )
    
))

fig.add_trace(go.Scatter(
    x = nbac_yearly["Year"],
    y = nbac_yearly["human_cause"],
    name = "Human Cause",
    
    customdata = (
        nbac_yearly["human_pct"]),


    hovertemplate = (
        "Year: %{x}"
        "<br>Count: %{y}"
        "<br>Human Cause: %{customdata:.2f} %"
    )
))

fig.add_trace(go.Scatter(
    x = nbac_yearly["Year"],
    y = nbac_yearly["undetermined_cause"],
    name = "Undetermined Cause",
    
    customdata = (
        nbac_yearly["undetermined_pct"]),


    hovertemplate = (
        "Year: %{x}"
        "<br>Count: %{y}"
        "<br>Undetermined Cause: %{customdata:.2f} %"
    )
))
fig.show()

In [38]:
nbac_yearly.head()

,Year,fires_n,rows_n,area_sum_ha,area_mean_ha,natural_cause,human_cause,undetermined_cause,natural_pct,human_pct,undetermined_pct
0,1990,509,512,8.587488e+05,1677.243712,229,94,189,44.990177,18.467583,37.131631
1,1991,575,582,1.530291e+06,2629.365928,203,154,225,35.304348,26.782609,39.130435
2,1992,331,333,8.640520e+05,2594.750768,137,82,114,41.389728,24.773414,34.441088
3,1993,377,378,1.947867e+06,5153.086565,164,56,158,43.501326,14.854111,41.909814
4,1994,708,720,5.073875e+06,7047.048654,333,78,309,47.033898,11.016949,43.644068


In [39]:
fig0 = go.Figure()

fig0.update_layout(
    title_text = "Canadian Wildfires Burned Area Over Time",
    xaxis_title = "Years",
    yaxis_title = "Area (ha)"
)

fig0.add_trace( go.Scatter(
    x = nbac_yearly["Year"],
    y = nbac_yearly["area_sum_ha"],
    name = "Total Burn Area",
    showlegend = True,

    # customdata = (np.column_stack([
    # ])),

    hovertemplate = (
        "Year: %{x}"
        "<br>Area: %{y}"
    )
))

fig0.add_trace(go.Scatter(
    x = nbac_yearly["Year"],
    y = nbac_yearly["area_mean_ha"],
    name = "Mean Fire Burn Area",

    hovertemplate= (
        "Year: %{x}"
        "<br>Area %{y}"
    )
))


### British Columbia

In [40]:

fig1= go.Figure()

fig1.update_layout(
    title_text = "British Columbia Wildfires and Causes Over Time",
    xaxis_title = "Years",
    yaxis_title = "Count"
)

fig1.add_trace( go.Scatter(
    x = bc_yearly["Year"],
    y = bc_yearly["fires_n"],
    name = "British Columbia Wildfires",
    showlegend = True,

    customdata = (np.column_stack([
        bc_yearly["natural_pct"],
        bc_yearly["human_pct"],
        bc_yearly["undetermined_pct"]
    ])),

    hovertemplate = (
        "Year: %{x}"
        "<br>Count: %{y}"
        "<br>Natural Cause: %{customdata[0]:.2f} %"
        "<br>Human Cause: %{customdata[1]:.2f} %"
        "<br>Undetermined Cause: %{customdata[2]:.2f} %"
    )

))

fig1.add_trace(go.Scatter(
    x = bc_yearly["Year"],
    y = bc_yearly['natural_cause'],
    name = "Natural Cause",

    customdata = (
        bc_yearly["natural_pct"]),


    hovertemplate = (
        "Year: %{x}"
        "<br>Count: %{y}"
        "<br>Natural Cause: %{customdata:.2f} %"
    )
    
))

fig1.add_trace(go.Scatter(
    x = bc_yearly["Year"],
    y = bc_yearly["human_cause"],
    name = "Human Cause",
    
    customdata = (
        bc_yearly["human_pct"]),


    hovertemplate = (
        "Year: %{x}"
        "<br>Count: %{y}"
        "<br>Human Cause: %{customdata:.2f} %"
    )
))

fig1.add_trace(go.Scatter(
    x = bc_yearly["Year"],
    y = bc_yearly["undetermined_cause"],
    name = "Undetermined Cause",
    
    customdata = (
        bc_yearly["undetermined_pct"]),


    hovertemplate = (
        "Year: %{x}"
        "<br>Count: %{y}"
        "<br>Undetermined Cause: %{customdata:.2f} %"
    )
))
fig1.show()

In [41]:
fig2 = go.Figure()

fig2.update_layout(
    title_text = "British Columbia Wildfires Burned Area Over Time",
    xaxis_title = "Years",
    yaxis_title = "Area (ha)"
)

fig2.add_trace( go.Scatter(
    x = bc_yearly["Year"],
    y = bc_yearly["area_sum_ha"],
    name = "Total Burn Area",
    showlegend = True,

    # customdata = (np.column_stack([
    # ])),

    hovertemplate = (
        "Year: %{x}"
        "<br>Area: %{y}"
    )
))

fig2.add_trace(go.Scatter(
    x = bc_yearly["Year"],
    y = bc_yearly["area_mean_ha"],
    name = "Mean Fire Burn Area",

    hovertemplate= (
        "Year: %{x}"
        "<br>Area %{y}"
    )
))


### Avalanche Canada

In [42]:

avcan_yearly = (
    avcan_stats
      .groupby("Year", dropna=False)
      .agg(
          fires_n=("Unique_gid_ID", "nunique"),        # unique fires
          rows_n=("Unique_gid_ID", "size"),            # number of records (optional)
          area_sum_ha=("Adjusted Burn Area (ha)", "sum"),     # total burned area
          area_mean_ha=("Adjusted Burn Area (ha)", "mean"),   # mean per-record area (see note below)
          natural_cause=("Is_Natural", "sum"),
          human_cause=("Is_Human", "sum"),
          undetermined_cause=("Is_Undetermined", "sum")
      )
      .reset_index()
)

avcan_yearly["natural_pct"] = (avcan_yearly["natural_cause"]/ avcan_yearly['fires_n'] ) * 100

avcan_yearly["human_pct"] = (avcan_yearly["human_cause"]/ avcan_yearly['fires_n']) * 100

avcan_yearly["undetermined_pct"] = (avcan_yearly["undetermined_cause"]/ avcan_yearly['fires_n'] ) * 100

In [43]:
avcan_yearly.head()

,Year,fires_n,rows_n,area_sum_ha,area_mean_ha,natural_cause,human_cause,undetermined_cause,natural_pct,human_pct,undetermined_pct
0,1990,52,53,8427.238571,159.004501,33,15,5,63.461538,28.846154,9.615385
1,1991,25,27,2256.738729,83.582916,4,23,0,16.000000,92.000000,0.000000
2,1992,46,46,8943.788935,194.430194,23,19,4,50.000000,41.304348,8.695652
3,1993,16,16,1212.621280,75.788830,3,12,1,18.750000,75.000000,6.250000
4,1994,71,72,10801.197910,150.016638,43,25,4,60.563380,35.211268,5.633803


In [44]:

fig3= go.Figure()

fig3.update_layout(
    title_text = "AvCan Wildfires and Causes Over Time",
    xaxis_title = "Years",
    yaxis_title = "Count"
)

fig3.add_trace( go.Scatter(
    x = avcan_yearly["Year"],
    y = avcan_yearly["fires_n"],
    name = "AvCan Wildfires",
    showlegend = True,

    customdata = (np.column_stack([
        avcan_yearly["natural_pct"],
        avcan_yearly["human_pct"],
        avcan_yearly["undetermined_pct"]
    ])),

    hovertemplate = (
        "Year: %{x}"
        "<br>Count: %{y}"
        "<br>Natural Cause: %{customdata[0]:.2f} %"
        "<br>Human Cause: %{customdata[1]:.2f} %"
        "<br>Undetermined Cause: %{customdata[2]:.2f} %"
    )

))

fig3.add_trace(go.Scatter(
    x = avcan_yearly["Year"],
    y = avcan_yearly['natural_cause'],
    name = "Natural Cause",

    customdata = (
        avcan_yearly["natural_pct"]),


    hovertemplate = (
        "Year: %{x}"
        "<br>Count: %{y}"
        "<br>Natural Cause: %{customdata:.2f} %"
    )
    
))

fig3.add_trace(go.Scatter(
    x = avcan_yearly["Year"],
    y = avcan_yearly["human_cause"],
    name = "Human Cause",
    
    customdata = (
        avcan_yearly["human_pct"]),


    hovertemplate = (
        "Year: %{x}"
        "<br>Count: %{y}"
        "<br>Human Cause: %{customdata:.2f} %"
    )
))

fig3.add_trace(go.Scatter(
    x = avcan_yearly["Year"],
    y = avcan_yearly["undetermined_cause"],
    name = "Undetermined Cause",
    
    customdata = (
        avcan_yearly["undetermined_pct"]),


    hovertemplate = (
        "Year: %{x}"
        "<br>Count: %{y}"
        "<br>Undetermined Cause: %{customdata:.2f} %"
    )
))
fig3.show()

In [45]:
fig4 = go.Figure()

fig4.update_layout(
    title_text = "AvCan Wildfires Burned Area Over Time",
    xaxis_title = "Years",
    yaxis_title = "Area (ha)"
)

fig4.add_trace( go.Scatter(
    x = avcan_yearly["Year"],
    y = avcan_yearly["area_sum_ha"],
    name = "Total Burn Area",
    showlegend = True,

    # customdata = (np.column_stack([
    # ])),

    hovertemplate = (
        "Year: %{x}"
        "<br>Area: %{y}"
    )
))

fig4.add_trace(go.Scatter(
    x = avcan_yearly["Year"],
    y = avcan_yearly["area_mean_ha"],
    name = "Mean Fire Burn Area",

    hovertemplate= (
        "Year: %{x}"
        "<br>Area %{y}"
    )
))


### Combined

In [46]:
fig6 = go.Figure()

fig6.update_layout(
    title_text = "Canadian Wildfires Burned Area Over Time",
    xaxis_title = "Years",
    yaxis_title = "Area (ha)"
)

fig6.add_trace( go.Scatter(
    x = nbac_yearly["Year"],
    y = nbac_yearly["area_sum_ha"],
    name = "Canada Burn Area",
    showlegend = True,

    # customdata = (np.column_stack([
    # ])),

    hovertemplate = (
        "Year: %{x}"
        "<br>Area: %{y}"
    )
))

fig6.add_trace( go.Scatter(
    x = bc_yearly["Year"],
    y = bc_yearly["area_sum_ha"],
    name = "BC Burn Area",
    showlegend = True,

    # customdata = (np.column_stack([
    # ])),

    hovertemplate = (
        "Year: %{x}"
        "<br>Area: %{y}"
    )
))

fig6.add_trace( go.Scatter(
    x = avcan_yearly["Year"],
    y = avcan_yearly["area_sum_ha"],
    name = "AvCan Burn Area",
    showlegend = True,

    # customdata = (np.column_stack([
    # ])),

    hovertemplate = (
        "Year: %{x}"
        "<br>Area: %{y}"
    )
))


In [47]:
# Dataset
min_year = int(NBAC_stats["Year"].min())
max_year = int(NBAC_stats["Year"].max())
n_years  = NBAC_stats["Year"].nunique()  # safest

# National Candian Fires
fires_total = NBAC_stats["Unique_gid_ID"].nunique()
avg_fires_per_year = fires_total / n_years

fires_per_year = NBAC_stats.groupby("Year")["Unique_gid_ID"].nunique()
median_fires_per_year = fires_per_year.median()

total_burn_ha = NBAC_stats["Adjusted Burn Area (ha)"].sum()
avg_burn_ha = total_burn_ha / n_years

burn_ha_per_year = NBAC_stats.groupby("Year")["Adjusted Burn Area (ha)"].sum()
median_burn_ha_per_year = burn_ha_per_year.median()
median_burn_km2_per_year = (median_burn_ha_per_year / 100)


total_burn_km2 = total_burn_ha / 100
avg_burn_km2 = avg_burn_ha / 100

largest_fire = NBAC_stats.sort_values("Adjusted Burn Area (ha)",ascending = False).iloc[0]
largest_burn_ha = largest_fire["Adjusted Burn Area (ha)"]
largest_burn_km = largest_burn_ha / 100
largest_fire_id = largest_fire["Unique_gid_ID"]
largeset_fire_prov = largest_fire["Province/Territory"]

# Per-fire total from summed pieces
burn_by_fire = NBAC_stats.groupby("Unique_gid_ID")["Adjusted Burn Area (ha)"].sum()



print(f"Years: {min_year}–{max_year} (n={n_years})")
print(f"Total fires: {fires_total:,}")
print(f"Avg fires/year: {avg_fires_per_year:,.0f}")
print(f"Median fires/year: {median_fires_per_year:,.0f}")
print("")



print(f"Total burned: {total_burn_ha:,.0f} ha ({total_burn_km2:,.0f} km²)")
print(f"Avg burned/year: {avg_burn_ha:,.0f} ha ({avg_burn_km2:,.0f} km²)")
print(f"Median burned/year: {median_burn_ha_per_year:,.0f} ha ({median_burn_km2_per_year:,.0f} km²)")
print("")

print(f"Largest fire ID: {largest_fire_id}. Province: {largeset_fire_prov}")
print(f"Largest fire burn: {burn_by_fire.max():,.0f} ha, ({(burn_by_fire.max()/100):,.0f} km²)")



Years: 1990–2024 (n=35)
Total fires: 39,616
Avg fires/year: 1,132
Median fires/year: 1,132

Total burned: 90,465,666 ha (904,657 km²)
Avg burned/year: 2,584,733 ha (25,847 km²)
Median burned/year: 1,796,630 ha (17,966 km²)

Largest fire ID: 2023_346. Province: QC
Largest fire burn: 1,146,937 ha, (11,469 km²)


In [48]:
NBAC_stats['Province/Territory'].unique()

array(['PC', 'NT', 'SK', 'MB', 'QC', 'AB', 'YT', 'ON', 'BC', 'NL', 'NS',
       'NB', 'NU'], dtype=object)

In [49]:
# AvCan fires
fires_total = avcan_stats["Unique_gid_ID"].nunique()
avg_fires_per_year = fires_total / n_years

fires_per_year = avcan_stats.groupby("Year")["Unique_gid_ID"].nunique()
median_fires_per_year = fires_per_year.median()

total_burn_ha = avcan_stats["Adjusted Burn Area (ha)"].sum()
avg_burn_ha = total_burn_ha / n_years

burn_ha_per_year = avcan_stats.groupby("Year")["Adjusted Burn Area (ha)"].sum()
median_burn_ha_per_year = burn_ha_per_year.median()
median_burn_km2_per_year = (median_burn_ha_per_year / 100)


total_burn_km2 = total_burn_ha / 100
avg_burn_km2 = avg_burn_ha / 100

largest_fire = avcan_stats.sort_values("Adjusted Burn Area (ha)",ascending = False).iloc[0]
largest_burn_ha = largest_fire["Adjusted Burn Area (ha)"]
largest_burn_km = largest_burn_ha / 100
largest_fire_id = largest_fire["Unique_gid_ID"]
largest_fire_prov = largest_fire["Province/Territory"]
largest_fire_region = avcan_stats[avcan_stats['Unique_gid_ID']=='2018_267']["Region"].iloc[0]

# Per-fire total from summed pieces
burn_by_fire = avcan_stats.groupby("Unique_gid_ID")["Adjusted Burn Area (ha)"].sum()


print(f"Total fires: {fires_total:,}")
print(f"Avg fires/year: {avg_fires_per_year:,.0f}")
print(f"Median fires/year: {median_fires_per_year:,.0f}")
print("")



print(f"Total burned: {total_burn_ha:,.0f} ha ({total_burn_km2:,.0f} km²)")
print(f"Avg burned/year: {avg_burn_ha:,.0f} ha ({avg_burn_km2:,.0f} km²)")
print(f"Median burned/year: {median_burn_ha_per_year:,.0f} ha ({median_burn_km2_per_year:,.0f} km²)")
print("")

print(f"Largest fire ID: {largest_fire_id}. Province: {largest_fire_prov}. AvCan Region: {largest_fire_region}")
print(f"Largest fire burn: {burn_by_fire.max():,.0f} ha, ({(burn_by_fire.max()/100):,.0f} km²)")


Total fires: 3,838
Avg fires/year: 110
Median fires/year: 79

Total burned: 2,271,966 ha (22,720 km²)
Avg burned/year: 64,913 ha (649 km²)
Median burned/year: 10,642 ha (106 km²)

Largest fire ID: 2018_267. Province: British Columbia. AvCan Region: Northwest_Inland
Largest fire burn: 126,578 ha, (1,266 km²)


In [50]:
stage_a_patches.columns.tolist()

['ee_feature_id',
 'Aspect Coherence (R)',
 'Majority Cardinal Direction',
 'Aspect Label',
 'Aspect Mean (deg)',
 'Max Elevation (m)',
 'Mean Elevation (m)',
 'Min Elevation (m)',
 'Elevation Relief (m)',
 'FireID',
 'Unique Fire ID (gid)',
 'National Park',
 'Patch Area (ha)',
 'Patch Area (m2)',
 'Patch ID',
 'Region',
 'Scenario',
 'Mean Slope Degree',
 'Slope Mean Percentage',
 'Slope Std Dev (deg)',
 'Subregion',
 'Year',
 'geometry',
 'Unique Patch ID']

In [58]:
avcan_regions.columns.tolist()

['Region', 'Subregion', 'Province/Territory', 'geometry']

In [63]:
print(f"Avcan regions in stage a: {stage_a_patches['Region'].nunique()}")
print(f"Avcan regions: {avcan_regions['Region'].nunique()}")
print(f"Avcan subregions: {avcan_regions['Subregion'].nunique()}")

Avcan regions in stage a: 22
Avcan regions: 22
Avcan subregions: 116


In [51]:
patches = stage_a_patches.groupby("Unique Fire ID (gid)")['Patch ID'].count()

total_patches = stage_a_patches['Unique Patch ID'].count()
total_fires = stage_a_patches['Unique Fire ID (gid)'].nunique()
total_fires_av = avcan_stats['Unique_gid_ID'].nunique()


avg_patches_per_fire_with_patches = total_patches / total_fires
avg_patches_per_avcan_fire = total_patches / total_fires_av

total_patches_area = stage_a_patches["Patch Area (ha)"].sum()
avg_patch_area = total_patches_area / total_patches

largest_patch = stage_a_patches.sort_values("Patch Area (ha)", ascending=False).iloc[0]

largest_patch_ha = largest_patch["Patch Area (ha)"]
largest_patch_km2 = largest_patch_ha / 100  # because 1 km² = 100 ha


largest_patch_id = largest_patch["Unique Patch ID"]
largest_patch_region = largest_patch["Region"]
largest_patch_subregion = largest_patch["Subregion"]

print(f'Total patches: {total_patches}')
print(f'Total fires with patches: {total_fires}')
print(f'Total avcan fires: {total_fires_av}')
print(f'Average number of patches per avcan fire: {avg_patches_per_avcan_fire:,.2f}')
print("")

print(f"Total patches area: {total_patches_area} ha")
print(f"Average Patch area: {avg_patch_area} ha")
print("")

print(f"Largest patch: ID {largest_patch_id} in Region: {largest_patch_region}")
print(f'Largest patch burn: {largest_patch_ha} ha ({largest_patch_km2} m2)')
print("")


Total patches: 6625
Total fires with patches: 1051
Total avcan fires: 3838
Average number of patches per avcan fire: 1.73

Total patches area: 377178.88 ha
Average Patch area: 56.932661132075474 ha

Largest patch: ID 2014_454_16.0 in Region: North_Rockies
Largest patch burn: 10075.89 ha (100.7589 m2)



In [52]:
NBAC_stats.columns.tolist()

['Unique_gid_ID',
 'Fire ID',
 'Year',
 'Province/Territory',
 'National Park',
 'Adjusted Burn Area (ha)',
 'Cause',
 'geometry',
 'Is_Natural',
 'Is_Human',
 'Is_Undetermined',
 'Province/Territory_full']

In [53]:
# ============================================================
# 0) Dataset window (NBAC)
# ============================================================
min_year = int(NBAC_stats["Year"].min())
max_year = int(NBAC_stats["Year"].max())
n_years  = int(NBAC_stats["Year"].nunique())

# ============================================================
# 1) NBAC national stats
#    NOTE: If NBAC rows can repeat per fire (e.g., split by prov/territory),
#    compute per-fire burn via groupby sum for "largest fire" and per-fire dist.
# ============================================================
nbac_fires_total = NBAC_stats["Unique_gid_ID"].nunique()
nbac_avg_fires_per_year = nbac_fires_total / n_years

nbac_fires_per_year = NBAC_stats.groupby("Year")["Unique_gid_ID"].nunique()
nbac_median_fires_per_year = float(nbac_fires_per_year.median())

nbac_total_burn_ha = float(NBAC_stats["Adjusted Burn Area (ha)"].sum())
nbac_avg_burn_ha_per_year = nbac_total_burn_ha / n_years
nbac_avg_fire_burn_ha = nbac_total_burn_ha / nbac_fires_total

nbac_burn_ha_per_year = NBAC_stats.groupby("Year")["Adjusted Burn Area (ha)"].sum()
nbac_median_burn_ha_per_year = float(nbac_burn_ha_per_year.median())

nbac_total_burn_km2 = nbac_total_burn_ha / 100
nbac_avg_burn_km2_per_year = nbac_avg_burn_ha_per_year / 100
nbac_median_burn_km2_per_year = nbac_median_burn_ha_per_year / 100

# Per-fire burn (safe against multi-row fires)
nbac_burn_by_fire = NBAC_stats.groupby("Unique_gid_ID")["Adjusted Burn Area (ha)"].sum()
nbac_largest_fire_id = nbac_burn_by_fire.idxmax()
nbac_largest_fire_burn_ha = float(nbac_burn_by_fire.max())
nbac_largest_fire_burn_km2 = nbac_largest_fire_burn_ha / 100

# Province for the largest fire: take the row with max area among that fire’s pieces
nbac_largest_fire_prov = (
    NBAC_stats.loc[NBAC_stats["Unique_gid_ID"] == nbac_largest_fire_id]
    .sort_values("Adjusted Burn Area (ha)", ascending=False)
    ["Province/Territory_full"]
    .iloc[0]
)
# Fire causes
# ============================================================
# Fire causes (NBAC) - per-fire, safe
# ============================================================
nbac_fire_id_col = "Unique_gid_ID" if "Unique_gid_ID" in NBAC_stats.columns else "Fire ID"
nbac_fires = NBAC_stats.drop_duplicates(subset=[nbac_fire_id_col]).copy()
nbac_n_fires = int(len(nbac_fires))

for col in ["Is_Natural", "Is_Human", "Is_Undetermined"]:
    if col in nbac_fires.columns:
        nbac_fires[col] = nbac_fires[col].fillna(0).astype(float)

nbac_pct_natural = float(nbac_fires["Is_Natural"].mean()) if "Is_Natural" in nbac_fires.columns else None
nbac_pct_human = float(nbac_fires["Is_Human"].mean()) if "Is_Human" in nbac_fires.columns else None
nbac_pct_undetermined = float(nbac_fires["Is_Undetermined"].mean()) if "Is_Undetermined" in nbac_fires.columns else None


# ============================================================
# 2) AvCan stats (same year window n_years from NBAC)
#    IMPORTANT: Use per-fire sums because AvCan has multiple rows per fire
# ============================================================
avcan_fire_area = (
    avcan_stats.groupby("Unique_gid_ID")["Adjusted Burn Area (ha)"]
    .sum()
    .sort_values(ascending=False)
)

avcan_total_fires = int(avcan_fire_area.size)
avcan_avg_fires_per_year = avcan_total_fires / n_years

avcan_fires_per_year = avcan_stats.groupby("Year")["Unique_gid_ID"].nunique()
avcan_median_fires_per_year = float(avcan_fires_per_year.median())

avcan_total_burn_ha = float(avcan_fire_area.sum())
avcan_avg_burn_ha_per_year = avcan_total_burn_ha / n_years

avcan_burn_ha_per_year = avcan_stats.groupby("Year")["Adjusted Burn Area (ha)"].sum()
avcan_median_burn_ha_per_year = float(avcan_burn_ha_per_year.median())

avcan_total_burn_km2 = avcan_total_burn_ha / 100
avcan_avg_burn_km2_per_year = avcan_avg_burn_ha_per_year / 100
avcan_median_burn_km2_per_year = avcan_median_burn_ha_per_year / 100

avcan_largest_fire_id = avcan_fire_area.index[0]
avcan_largest_fire_burn_ha = float(avcan_fire_area.iloc[0])
avcan_largest_fire_burn_km2 = avcan_largest_fire_burn_ha / 100

# Province and (a) region(s) for that fire
avcan_largest_fire_prov = (
    avcan_stats.loc[avcan_stats["Unique_gid_ID"] == avcan_largest_fire_id, "Province/Territory"]
    .dropna()
    .mode()
    .iloc[0]
)
# If a fire spans multiple AvCan regions, keep them all (sorted unique)
largest_fire_mask = avcan_stats["Unique_gid_ID"].eq(avcan_largest_fire_id)

avcan_largest_fire_regions = (
    avcan_stats.loc[largest_fire_mask, "Region"]
    .dropna()
    .astype(str)
    .sort_values()
    .unique()
    .tolist()
)

# Fire causes
# ----------------------------
# 1) Use 1 row per fire (prevents double-counting)
# ----------------------------
avcan_fire_id_col = "Unique_gid_ID" if "Unique_gid_ID" in avcan_stats.columns else "Fire ID"
avcan_fires = avcan_stats.drop_duplicates(subset=[avcan_fire_id_col]).copy()
avcan_n_fires = int(len(avcan_fires))

for col in ["Is_Natural", "Is_Human", "Is_Undetermined"]:
    if col in avcan_fires.columns:
        avcan_fires[col] = avcan_fires[col].fillna(0).astype(float)
# ----------------------------
# 2) Percentages from your boolean flags (if present)
#    (expects 0/1 or True/False)
# ----------------------------
avcan_pct_natural = float(avcan_fires["Is_Natural"].mean()) if "Is_Natural" in avcan_fires.columns else None
avcan_pct_human = float(avcan_fires["Is_Human"].mean()) if "Is_Human" in avcan_fires.columns else None
avcan_pct_undetermined = float(avcan_fires["Is_Undetermined"].mean()) if "Is_Undetermined" in avcan_fires.columns else None





# FIX: use avcan_fires (DF), not avcan_n_fires (int)

# ============================================================
# 3) Stage A patches (dedupe by Unique Patch ID)
# ============================================================
patches_u = stage_a_patches.drop_duplicates("Unique Patch ID").copy()

stagea_total_patches = int(patches_u["Unique Patch ID"].nunique())
stagea_fires_with_patches = int(patches_u["Unique Fire ID (gid)"].nunique())

# Average patches per fire (fires with patches) and per AvCan fire
patches_per_fire_with_patches = patches_u.groupby("Unique Fire ID (gid)")["Unique Patch ID"].nunique()
stagea_avg_patches_per_fire_with_patches = float(patches_per_fire_with_patches.mean())
stagea_median_patches_per_fire_with_patches = float(patches_per_fire_with_patches.median())
stagea_p90_patches_per_fire_with_patches = float(patches_per_fire_with_patches.quantile(0.90))

stagea_avg_patches_per_avcan_fire = stagea_total_patches / avcan_total_fires

# Patch area totals + distribution
stagea_total_patch_ha = float(patches_u["Patch Area (ha)"].sum())
stagea_avg_patch_ha = stagea_total_patch_ha / stagea_total_patches
stagea_median_patch_ha = float(patches_u["Patch Area (ha)"].median())
stagea_p90_patch_ha = float(patches_u["Patch Area (ha)"].quantile(0.90))
stagea_p95_patch_ha = float(patches_u["Patch Area (ha)"].quantile(0.95))

# Largest patch
largest_patch = patches_u.sort_values("Patch Area (ha)", ascending=False).iloc[0]
stagea_largest_patch_id = largest_patch["Unique Patch ID"]
stagea_largest_patch_region = largest_patch["Region"]
stagea_largest_patch_subregion = largest_patch["Subregion"]
stagea_largest_patch_ha = float(largest_patch["Patch Area (ha)"])
stagea_largest_patch_km2 = stagea_largest_patch_ha / 100

# Coverage / intensity (Stage A vs AvCan)
stagea_pct_avcan_fires_with_patches = stagea_fires_with_patches / avcan_total_fires
stagea_pct_avcan_burn_in_patches = stagea_total_patch_ha / avcan_total_burn_ha
stagea_avg_patch_ha_per_avcan_fire = stagea_total_patch_ha / avcan_total_fires
stagea_avg_patch_ha_per_fire_with_patches = stagea_total_patch_ha / stagea_fires_with_patches
stagea_patches_per_1000ha_burned = stagea_total_patches / (avcan_total_burn_ha / 1000)

# Temporal peak patch year
stagea_patch_by_year = patches_u.groupby("Year")["Patch Area (ha)"].sum()
stagea_max_patch_year = int(stagea_patch_by_year.idxmax())
stagea_max_patch_year_ha = float(stagea_patch_by_year.loc[stagea_max_patch_year])

# Spatial: top region by patch ha
stagea_patch_by_region = patches_u.groupby("Region")["Patch Area (ha)"].sum().sort_values(ascending=False)
stagea_top_region = stagea_patch_by_region.index[0]
stagea_top_region_patch_ha = float(stagea_patch_by_region.iloc[0])
stagea_top_region_patch_share = stagea_top_region_patch_ha / stagea_total_patch_ha

# Aspect / terrain rollups
stagea_pct_patches_mixed_aspect = float((patches_u["Aspect Label"].eq("Mixed")).mean())
stagea_aspect_R_median = float(patches_u["Aspect Coherence (R)"].median())
stagea_slope_mean_deg_median = float(patches_u["Mean Slope Degree"].median())
stagea_elev_mean_m_median = float(patches_u["Mean Elevation (m)"].median())

# ============================================================
# 4) Human-readable console output
# ============================================================
print(f"Years: {min_year}–{max_year} (n={n_years})\n")

print("=== NBAC (National) ===")
print(f"Total fires: {nbac_fires_total:,}")
print(f"Avg fires/year: {nbac_avg_fires_per_year:,.0f}")
print(f"Median fires/year: {nbac_median_fires_per_year:,.0f}\n")

print(f"Total burned: {nbac_total_burn_ha:,.0f} ha ({nbac_total_burn_km2:,.0f} km²)")
print(f"Avg burned/year: {nbac_avg_burn_ha_per_year:,.0f} ha ({nbac_avg_burn_km2_per_year:,.0f} km²)")
print(f"Median burned/year: {nbac_median_burn_ha_per_year:,.0f} ha ({nbac_median_burn_km2_per_year:,.0f} km²)\n")

print(f"Largest fire ID: {nbac_largest_fire_id}. Province: {nbac_largest_fire_prov}")
print(f"Largest fire burn: {nbac_largest_fire_burn_ha:,.0f} ha ({nbac_largest_fire_burn_km2:,.0f} km²)\n")

print("=== AvCan (NBAC intersect AvCan regions) ===")
print(f"Total fires: {avcan_total_fires:,}")
print(f"Avg fires/year: {avcan_avg_fires_per_year:,.0f}")
print(f"Median fires/year: {avcan_median_fires_per_year:,.0f}\n")

print(f"Total burned: {avcan_total_burn_ha:,.0f} ha ({avcan_total_burn_km2:,.0f} km²)")
print(f"Avg burned/year: {avcan_avg_burn_ha_per_year:,.0f} ha ({avcan_avg_burn_km2_per_year:,.0f} km²)")
print(f"Median burned/year: {avcan_median_burn_ha_per_year:,.0f} ha ({avcan_median_burn_km2_per_year:,.0f} km²)\n")

print(f"Largest fire ID: {avcan_largest_fire_id}. Province: {avcan_largest_fire_prov}. AvCan Regions: {', '.join(avcan_largest_fire_regions)}")
print(f"Largest fire burn: {avcan_largest_fire_burn_ha:,.0f} ha ({avcan_largest_fire_burn_km2:,.0f} km²)\n")

print("=== Stage A (Severe patches) ===")
print(f"Total patches (unique): {stagea_total_patches:,}")
print(f"Total fires with patches: {stagea_fires_with_patches:,}")
print(f"Pct of AvCan fires with patches: {stagea_pct_avcan_fires_with_patches:.1%}")
print(f"Avg patches per fire (fires with patches): {stagea_avg_patches_per_fire_with_patches:,.2f}")
print(f"Median patches per fire (fires with patches): {stagea_median_patches_per_fire_with_patches:,.0f}")
print(f"Avg patches per AvCan fire (all fires): {stagea_avg_patches_per_avcan_fire:,.2f}\n")

print(f"Total patch area: {stagea_total_patch_ha:,.2f} ha ({stagea_total_patch_ha/100:,.2f} km²)")
print(f"Avg patch area: {stagea_avg_patch_ha:,.2f} ha")
print(f"Median patch area: {stagea_median_patch_ha:,.2f} ha (p90={stagea_p90_patch_ha:,.2f}, p95={stagea_p95_patch_ha:,.2f})")
print(f"Pct of AvCan burn area in patches: {stagea_pct_avcan_burn_in_patches:.1%}")
print(f"Patches per 1,000 ha burned: {stagea_patches_per_1000ha_burned:,.2f}\n")

print(f"Largest patch: ID {stagea_largest_patch_id} in {stagea_largest_patch_region} / {stagea_largest_patch_subregion}")
print(f"Largest patch area: {stagea_largest_patch_ha:,.2f} ha ({stagea_largest_patch_km2:,.2f} km²)\n")

print(f"Patch peak year: {stagea_max_patch_year} ({stagea_max_patch_year_ha:,.0f} ha)")
print(f"Top region by patch area: {stagea_top_region} ({stagea_top_region_patch_share:.1%} of patch area)\n")

print(f"Pct patches labeled Mixed aspect: {stagea_pct_patches_mixed_aspect:.1%}")
print(f"Median aspect coherence R: {stagea_aspect_R_median:.2f}")
print(f"Median mean slope (deg): {stagea_slope_mean_deg_median:.1f}")
print(f"Median mean elevation (m): {stagea_elev_mean_m_median:,.0f}")

# ============================================================
# 5) Dict output (for YAML dumping)
# ============================================================
summary_additions = {
    "data": {
        "min_year": min_year,
        "max_year": max_year,
        "n_years": n_years,
    },
    "nbac": {
        "total_fires": int(nbac_fires_total),
        "avg_fires_per_year": float(nbac_avg_fires_per_year),
        "median_fires_per_year": float(nbac_median_fires_per_year),
        "total_burn_ha": float(nbac_total_burn_ha),
        "total_burn_km2": float(nbac_total_burn_km2),
        "avg_burn_ha_per_year": float(nbac_avg_burn_ha_per_year),
        "avg_burn_km2_per_year": float(nbac_avg_burn_km2_per_year),
        "avg_fire_burn_ha": float(nbac_avg_fire_burn_ha),
        "avg_fire_burn_km2": float(nbac_avg_fire_burn_ha)/100,
        "median_burn_ha_per_year": float(nbac_median_burn_ha_per_year),
        "median_burn_km2_per_year": float(nbac_median_burn_km2_per_year),
        "largest_fire_id": str(nbac_largest_fire_id),
        "largest_fire_province": str(nbac_largest_fire_prov),
        "largest_fire_burn_ha": float(nbac_largest_fire_burn_ha),
        "largest_fire_burn_km2": float(nbac_largest_fire_burn_km2),
        # Causes
        "natural_cause": float(nbac_pct_natural),
        "human_cause": float(nbac_pct_human),
        "undetermined_cause": float(nbac_pct_undetermined),
    },
    "avcan": {
        "total_fires": int(avcan_total_fires),
        "avg_fires_per_year": float(avcan_avg_fires_per_year),
        "median_fires_per_year": float(avcan_median_fires_per_year),
        "total_burn_ha": float(avcan_total_burn_ha),
        "total_burn_km2": float(avcan_total_burn_km2),
        "avg_burn_ha_per_year": float(avcan_avg_burn_ha_per_year),
        "avg_burn_km2_per_year": float(avcan_avg_burn_km2_per_year),
        "median_burn_ha_per_year": float(avcan_median_burn_ha_per_year),
        "median_burn_km2_per_year": float(avcan_median_burn_km2_per_year),
        "largest_fire_id": str(avcan_largest_fire_id),
        "largest_fire_province": str(avcan_largest_fire_prov),
        "largest_fire_regions": avcan_largest_fire_regions,
        "largest_fire_burn_ha": float(avcan_largest_fire_burn_ha),
        "largest_fire_burn_km2": float(avcan_largest_fire_burn_km2),
        # Distribution / concentration add-ons
        "median_burn_per_fire_ha": float(avcan_fire_area.median()),
        "median_burn_per_fire_km2": float(avcan_fire_area.median())/100,
        "avg_burn_ha_per_fire_ha": float(avcan_fire_area.mean()),
        "avg_burn_ha_per_fire_km2": float(avcan_fire_area.mean())/100,
        "p90_burn_ha_per_fire": float(avcan_fire_area.quantile(0.90)),
        "p95_burn_ha_per_fire": float(avcan_fire_area.quantile(0.95)),
        # Causes
        "natural_cause": float(avcan_pct_natural),
        "human_cause": float(avcan_pct_human),
        "undetermined_cause": float(avcan_pct_undetermined),


    },
    "stage_a": {
        "total_patches": int(stagea_total_patches),
        "total_fires_with_patches": int(stagea_fires_with_patches),
        "avg_patches_per_fire_with_patches": float(stagea_avg_patches_per_fire_with_patches),
        "median_patches_per_fire_with_patches": float(stagea_median_patches_per_fire_with_patches),
        "p90_patches_per_fire_with_patches": float(stagea_p90_patches_per_fire_with_patches),
        "avg_patches_per_avcan_fire": float(stagea_avg_patches_per_avcan_fire),
        "total_patch_ha": float(stagea_total_patch_ha),
        "avg_patch_ha": float(stagea_avg_patch_ha),
        "median_patch_ha": float(stagea_median_patch_ha),
        "p90_patch_ha": float(stagea_p90_patch_ha),
        "p95_patch_ha": float(stagea_p95_patch_ha),
        "largest_patch_id": str(stagea_largest_patch_id),
        "largest_patch_region": str(stagea_largest_patch_region),
        "largest_patch_subregion": str(stagea_largest_patch_subregion),
        "largest_patch_ha": float(stagea_largest_patch_ha),
        "largest_patch_km2": float(stagea_largest_patch_km2),
        # Coverage / intensity add-ons
        "pct_avcan_fires_with_patches": float(stagea_pct_avcan_fires_with_patches),
        "pct_avcan_burn_in_patches": float(stagea_pct_avcan_burn_in_patches),
        "avg_patch_ha_per_avcan_fire": float(stagea_avg_patch_ha_per_avcan_fire),
        "avg_patch_ha_per_fire_with_patches": float(stagea_avg_patch_ha_per_fire_with_patches),
        "patches_per_1000ha_burned": float(stagea_patches_per_1000ha_burned),
        "max_patch_year": int(stagea_max_patch_year),
        "max_patch_year_ha": float(stagea_max_patch_year_ha),
        "top_region_by_patch_ha": str(stagea_top_region),
        "top_region_patch_share": float(stagea_top_region_patch_share),
        # Terrain / aspect add-ons
        "pct_patches_mixed_aspect": float(stagea_pct_patches_mixed_aspect),
        "aspect_R_median": float(stagea_aspect_R_median),
        "slope_mean_deg_median": float(stagea_slope_mean_deg_median),
        "elev_mean_m_median": float(stagea_elev_mean_m_median),
    },
}

summary_additions


Years: 1990–2024 (n=35)

=== NBAC (National) ===
Total fires: 39,616
Avg fires/year: 1,132
Median fires/year: 1,132

Total burned: 90,465,666 ha (904,657 km²)
Avg burned/year: 2,584,733 ha (25,847 km²)
Median burned/year: 1,796,630 ha (17,966 km²)

Largest fire ID: 2023_310. Province: Northwest Territories
Largest fire burn: 1,146,937 ha (11,469 km²)

=== AvCan (NBAC intersect AvCan regions) ===
Total fires: 3,838
Avg fires/year: 110
Median fires/year: 79

Total burned: 2,271,966 ha (22,720 km²)
Avg burned/year: 64,913 ha (649 km²)
Median burned/year: 10,642 ha (106 km²)

Largest fire ID: 2024_451. Province: Alberta. AvCan Regions: Jasper
Largest fire burn: 126,578 ha (1,266 km²)

=== Stage A (Severe patches) ===
Total patches (unique): 6,625
Total fires with patches: 1,051
Pct of AvCan fires with patches: 27.4%
Avg patches per fire (fires with patches): 6.30
Median patches per fire (fires with patches): 2
Avg patches per AvCan fire (all fires): 1.73

Total patch area: 377,178.88 ha (3

{'data': {'min_year': 1990, 'max_year': 2024, 'n_years': 35},
 'nbac': {'total_fires': 39616,
  'avg_fires_per_year': 1131.8857142857144,
  'median_fires_per_year': 1132.0,
  'total_burn_ha': 90465665.73580533,
  'total_burn_km2': 904656.6573580534,
  'avg_burn_ha_per_year': 2584733.3067372954,
  'avg_burn_km2_per_year': 25847.333067372954,
  'avg_fire_burn_ha': 2283.563856416734,
  'avg_fire_burn_km2': 22.83563856416734,
  'median_burn_ha_per_year': 1796630.04859189,
  'median_burn_km2_per_year': 17966.3004859189,
  'largest_fire_id': '2023_310',
  'largest_fire_province': 'Northwest Territories',
  'largest_fire_burn_ha': 1146936.731336,
  'largest_fire_burn_km2': 11469.367313359999,
  'natural_cause': 0.544906098546042,
  'human_cause': 0.2753937802907916,
  'undetermined_cause': 0.1797001211631664},
 'avcan': {'total_fires': 3838,
  'avg_fires_per_year': 109.65714285714286,
  'median_fires_per_year': 79.0,
  'total_burn_ha': 2271966.2429026375,
  'total_burn_km2': 22719.66242902637

In [54]:

# If you have numpy / pandas scalars in the dict, this keeps YAML clean
def _to_builtin(x):
    try:
        import numpy as np
        import pandas as pd
        if isinstance(x, (np.integer,)):
            return int(x)
        if isinstance(x, (np.floating,)):
            return float(x)
        if isinstance(x, (np.bool_,)):
            return bool(x)
        if isinstance(x, (pd.Timestamp,)):
            return x.isoformat()
    except Exception:
        pass

    if isinstance(x, dict):
        return {k: _to_builtin(v) for k, v in x.items()}
    if isinstance(x, (list, tuple)):
        return [_to_builtin(v) for v in x]
    return x

# -------------------------------------------------------------------
# Write summary_additions -> REPO_ROOT/streamlit_app/config/summary_stats.yaml
# -------------------------------------------------------------------
out_path = Path(REPO_ROOT) / "streamlit_app" / "config" / "summary_stats.yaml"
out_path.parent.mkdir(parents=True, exist_ok=True)

with out_path.open("w", encoding="utf-8") as f:
    yaml.safe_dump(
        _to_builtin(summary_additions),
        f,
        sort_keys=False,
        default_flow_style=False,
        allow_unicode=True,
    )

print(f"Wrote: {out_path}")


Wrote: /Users/mitchellpalmer/Projects/canada-wildfire-avcan/streamlit_app/config/summary_stats.yaml


In [55]:
avcan_gids  = set(avcan_fire_area.index.astype(str))
patch_gids  = set(patches_u["Unique Fire ID (gid)"].dropna().astype(str).unique())

missing_in_avcan = sorted(patch_gids - avcan_gids)[:10]
missing_in_patches = sorted(avcan_gids - patch_gids)
len(missing_in_avcan), len(missing_in_patches)


(0, 2787)